In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, math, glob, random
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils as vutils
from torchvision.transforms import functional as TF
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
# =========================
# Dataset (pixel space @ 512×512, no VAE/no Diffusers)
# =========================

class OutpaintDataset(Dataset):
    """
    Reads RGB images from root_dir, applies random rotations, creates a random rectangular
    mask (bbox) and returns (masked_img, full_img, bbox_tensor[4]).
    Images are normalized to [-1, 1]. Masked area is zeroed out in masked_img.
    """
    def __init__(self, root_dir, img_size=512, masks_per_image=40):
        self.img_files = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                          if f.lower().endswith((".png",".jpg",".jpeg"))]
        self.img_size = img_size
        self.masks_per_image = masks_per_image
        self.transform = transforms.Compose([
            transforms.RandomChoice([
                transforms.Lambda(lambda x: x),
                transforms.Lambda(lambda x: TF.rotate(x, 90)),
                transforms.Lambda(lambda x: TF.rotate(x, 180)),
                transforms.Lambda(lambda x: TF.rotate(x, 270)),
            ]),
            transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),  # -> [-1,1]
        ])

    def __len__(self):
        return len(self.img_files) * self.masks_per_image

    def _gen_random_bbox(self):
        """Generate biased random bounding boxes with extreme or moderate aspect ratios.
        Returns normalized (x1, y1, x2, y2) in [0,1]."""
        if random.random() < 0.5:
            aspect_ratio = random.uniform(1, 33)
        else:
            aspect_ratio = random.uniform(0.03, 1)
        area = random.uniform(0.1, 0.5)**2
        w = min(np.sqrt(area * aspect_ratio), 0.99)
        h = min(np.sqrt(area / aspect_ratio), 0.99)
        x1 = random.uniform(0.02, 0.98 - w)
        y1 = random.uniform(0.02, 0.98 - h)
        return (float(x1), float(y1), float(x1 + w), float(y1 + h))

    def __getitem__(self, idx):
        img_idx = idx // self.masks_per_image
        img = Image.open(self.img_files[img_idx]).convert("RGB")
        img = self.transform(img)  # [3, H, W], in [-1,1]

        # Random bbox & mask
        x1, y1, x2, y2 = self._gen_random_bbox()
        C, H, W = img.shape
        xi1 = int(x1 * W); yi1 = int(y1 * H)
        xi2 = int(x2 * W); yi2 = int(y2 * H)
        xi1, yi1 = max(0, xi1), max(0, yi1)
        xi2, yi2 = min(W, xi2), min(H, yi2)

        mask = torch.zeros(1, H, W, dtype=img.dtype)
        # NOTE: correct indexing: [:, y1:y2, x1:x2]
        if yi2 > yi1 and xi2 > xi1:
            mask[:, yi1:yi2, xi1:xi2] = 1.0

        # Apply mask (zero out region)
        masked_img = img * (1 - mask)

        bbox = torch.tensor([x1, y1, x2, y2], dtype=torch.float32)
        return masked_img, img, bbox, mask

In [ ]:


# =========================
# Utils
# =========================

def to_uint8(img_tensor):
    """[-1,1] -> uint8 image tensor (H,W,3)."""
    img = (img_tensor.clamp(-1, 1) + 1) * 0.5
    img = (img * 255.0).round().clamp(0, 255).to(torch.uint8)
    return img

_SOBEL_X = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
_SOBEL_Y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)

def grad_mag(x):  # x: [-1,1], [B,3,H,W]
    kx = _SOBEL_X.to(x.device).repeat(3,1,1,1)
    ky = _SOBEL_Y.to(x.device).repeat(3,1,1,1)
    gx = F.conv2d(x, kx, padding=1, groups=3)
    gy = F.conv2d(x, ky, padding=1, groups=3)
    return torch.sqrt(gx*gx + gy*gy + 1e-8)


def _gn_groups(c):
    g = min(32, c)
    while c % g != 0 and g > 1:
        g //= 2
    return g



# =========================
# Time Embedding & Blocks
# =========================

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
    def forward(self, t):
        device = t.device; half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(0, half, device=device) / (half - 1 + 1e-8))
        args = t[:, None].float() * freqs[None]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return F.pad(emb, (0,1)) if self.dim % 2 == 1 else emb

class TimeMLP(nn.Module):
    def __init__(self, dim, out):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, out), nn.SiLU(), nn.Linear(out, out))
    def forward(self, t_emb): return self.net(t_emb)

def conv3x3(in_c, out_c):
    return nn.Conv2d(in_c, out_c, 3, padding=1)

class ResBlock(nn.Module):
    def __init__(self, in_c, out_c, time_dim):
        super().__init__()
        g1 = _gn_groups(in_c); g2 = _gn_groups(out_c)
        self.norm1 = nn.GroupNorm(g1, in_c); self.act1 = nn.SiLU(); self.conv1 = conv3x3(in_c, out_c)
        self.norm2 = nn.GroupNorm(g2, out_c); self.act2 = nn.SiLU(); self.conv2 = conv3x3(out_c, out_c)
        self.time  = nn.Linear(time_dim, out_c)
        self.skip  = nn.Identity() if in_c == out_c else nn.Conv2d(in_c, out_c, 1)
    def forward(self, x, t_emb):
        h = self.conv1(self.act1(self.norm1(x)))
        h = h + self.time(t_emb)[:, :, None, None]
        h = self.conv2(self.act2(self.norm2(h)))
        return h + self.skip(x)

class ResStack(nn.Module):
    def __init__(self, blocks):
        super().__init__(); self.blocks = nn.ModuleList(blocks)
    def forward(self, x, t_emb):
        for b in self.blocks:
            x = b(x, t_emb)
        return x

class Down(nn.Module):
    def __init__(self, c):
        super().__init__(); self.op = nn.Conv2d(c, c, 4, stride=2, padding=1)
    def forward(self, x): return self.op(x)

class Up(nn.Module):
    def __init__(self, c):
        super().__init__(); self.op = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"), conv3x3(c, c))
    def forward(self, x): return self.op(x)

# =========================
# Condition Encoder & Cross-Attn & High-Res Fuse
# =========================

class ConvGNAct(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.norm = nn.GroupNorm(_gn_groups(in_c), in_c)
        self.act  = nn.SiLU()
        self.conv = nn.Conv2d(in_c, out_c, 3, padding=1)
    def forward(self, x): return self.conv(self.act(self.norm(x)))

class CondEncoder(nn.Module):
    def __init__(self, in_ch=3, chans=(64,128,256,512)):
        super().__init__()
        self.stem = nn.Conv2d(in_ch, chans[0], 3, padding=1)
        downs, blocks = [], []
        in_c = chans[0]
        for i, c in enumerate(chans):
            blocks.append(nn.Sequential(ConvGNAct(in_c, c), ConvGNAct(c, c)))
            in_c = c
            if i < len(chans) - 1:
                downs.append(nn.Conv2d(in_c, in_c, 4, stride=2, padding=1))
        self.blocks = nn.ModuleList(blocks); self.downs = nn.ModuleList(downs)
    def forward(self, x):
        feats = []
        x = self.stem(x)
        for i, blk in enumerate(self.blocks):
            x = blk(x); feats.append(x)
            if i < len(self.blocks) - 1: x = self.downs[i](x)
        return feats

class CrossAttention2D(nn.Module):
    def __init__(self, dim_q, dim_kv, heads=4, dropout=0.0):
        super().__init__()
        cand = [8,6,4,3,2,1]
        self.heads = next((h for h in cand if dim_q % h == 0), 1)
        self.scale = (dim_q // self.heads) ** -0.5
        self.to_q = nn.Conv2d(dim_q, dim_q, 1, bias=False)
        self.to_k = nn.Conv2d(dim_kv, dim_q, 1, bias=False)
        self.to_v = nn.Conv2d(dim_kv, dim_q, 1, bias=False)
        self.proj = nn.Conv2d(dim_q, dim_q, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, context):
        B, C, H, W = x.shape
        q = self.to_q(x); k = self.to_k(context); v = self.to_v(context)
        h = self.heads
        def split(t): return t.view(B, h, C//h, H*W).permute(0,1,3,2)
        q, k, v = split(q), split(k), split(v)
        attn = (q @ k.transpose(-2,-1)) * self.scale
        attn = self.drop(attn.softmax(dim=-1))
        out = attn @ v
        out = out.permute(0,1,3,2).contiguous().view(B, C, H, W)
        return self.proj(out)

class HighResFuse(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.dw = nn.Conv2d(2*c, 2*c, 3, padding=1, groups=2*c)  # depthwise
        self.pw = nn.Conv2d(2*c, c, 1)                           # pointwise
        self.norm = nn.GroupNorm(_gn_groups(c), c)
        self.act  = nn.SiLU()
        hidden = max(8, c // 16)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c, hidden, 1), nn.SiLU(),
            nn.Conv2d(hidden, c, 1), nn.Sigmoid()
        )
    def forward(self, x, cond_feat):
        h = torch.cat([x, cond_feat], dim=1)
        h = self.dw(h)
        h = self.pw(h)
        h = self.act(self.norm(h))
        h = h * self.se(h)
        return x + h

# =========================
# Conditional UNet
# =========================

class ConditionalUNet(nn.Module):
    def __init__(self,
                 base=128, ch_mult=(1,2,4,8), width_scale=1.25,
                 blocks_per_stage=3, time_dim=320,
                 in_channels=6, out_channels=3,
                 in_res=512,
                 attn_res={64},
                 highres_res={128,256,512},
                 attn_heads=4, attn_dropout=0.0):
        super().__init__()
        self.attn_res    = set(attn_res)
        self.highres_res = set(highres_res)

        self.time_mlp = nn.Sequential(SinusoidalPosEmb(time_dim), TimeMLP(time_dim, time_dim))

        chans = [int(base * width_scale) * m for m in ch_mult]
        self.chans = chans
        self.res_list = [in_res // (2**i) for i in range(len(chans))]  # e.g. [512,256,128,64]

        self.cond_enc = CondEncoder(in_ch=3, chans=chans)
        self.in_conv = conv3x3(in_channels, chans[0])

        # Down
        self.downs = nn.ModuleList(); self.res_down = nn.ModuleList()
        self.ca_down = nn.ModuleList(); self.proj_down = nn.ModuleList(); self.fuse_down = nn.ModuleList()

        in_c = chans[0]
        for i, c in enumerate(chans):
            blocks = [ResBlock(in_c, c, time_dim)]
            for _ in range(blocks_per_stage - 1):
                blocks.append(ResBlock(c, c, time_dim))
            self.res_down.append(ResStack(blocks))

            res_here = self.res_list[i]
            self.ca_down.append(CrossAttention2D(dim_q=c, dim_kv=c, heads=attn_heads, dropout=attn_dropout))
            self.proj_down.append(nn.Conv2d(c, c, 1))
            self.fuse_down.append(HighResFuse(c) if res_here in self.highres_res else nn.Identity())

            in_c = c
            if i < len(chans)-1:
                self.downs.append(Down(in_c))

        # Mid
        self.mid = ResStack([ResBlock(in_c, in_c, time_dim), ResBlock(in_c, in_c, time_dim)])
        self.ca_mid   = CrossAttention2D(dim_q=in_c, dim_kv=in_c, heads=attn_heads, dropout=0.0)
        self.proj_mid = nn.Conv2d(in_c, in_c, 1)
        self.fuse_mid = HighResFuse(in_c) if (self.res_list[-1] in self.highres_res) else nn.Identity()

        # Up
        self.ups = nn.ModuleList(); self.res_up = nn.ModuleList()
        self.ca_up = nn.ModuleList(); self.proj_up = nn.ModuleList(); self.fuse_up = nn.ModuleList()

        for i in reversed(range(len(chans))):
            c = chans[i]
            blocks = [ResBlock(in_c + c, c, time_dim)]
            for _ in range(blocks_per_stage - 1):
                blocks.append(ResBlock(c, c, time_dim))
            self.res_up.append(ResStack(blocks))

            res_here = self.res_list[i]
            self.ca_up.append(CrossAttention2D(dim_q=c, dim_kv=c, heads=attn_heads, dropout=attn_dropout))
            self.proj_up.append(nn.Conv2d(c, c, 1))
            self.fuse_up.append(HighResFuse(c) if res_here in self.highres_res else nn.Identity())

            in_c = c
            if i > 0:
                self.ups.append(Up(in_c))

        self.out = nn.Sequential(nn.GroupNorm(_gn_groups(in_c), in_c), nn.SiLU(), conv3x3(in_c, out_channels))

    def _resize_like(self, feat, ref):
        if feat.shape[2:] != ref.shape[2:]:
            feat = F.interpolate(feat, size=ref.shape[2:], mode="nearest")
        return feat

    def _inject(self, x, cond_feat, res_here, attn_module, proj_module, fuse_module):
        cond_feat = self._resize_like(cond_feat, x)
        if (res_here in self.attn_res):
            return x + attn_module(x, cond_feat)
        if isinstance(fuse_module, HighResFuse):
            return fuse_module(x, cond_feat)
        return x + proj_module(cond_feat)

    def forward(self, x_noisy, t, cond):
        t_emb = self.time_mlp(t)
        cond_feats = self.cond_enc(cond)

        # Input concat: [x_t, cond]
        x = torch.cat([x_noisy, cond], dim=1)
        x = self.in_conv(x)

        feats = []
        for i, (blk, attn, proj, fuse, down) in enumerate(zip(
            self.res_down, self.ca_down, self.proj_down, self.fuse_down, list(self.downs)+[None]
        )):
            x = blk(x, t_emb)
            x = self._inject(x, cond_feats[i], self.res_list[i], attn, proj, fuse)
            feats.append(x)
            if down is not None: x = down(x)

        x = self.mid(x, t_emb)
        x = self._inject(x, cond_feats[-1], self.res_list[-1], self.ca_mid, self.proj_mid, self.fuse_mid)

        for i, (blk, attn, proj, fuse, up) in enumerate(zip(
            self.res_up, self.ca_up, self.proj_up, self.fuse_up, list(self.ups)+[None]
        )):
            skip = feats.pop()
            x = torch.cat([x, skip], dim=1)
            x = blk(x, t_emb)
            level_idx = len(self.chans) - 1 - i
            x = self._inject(x, cond_feats[level_idx], self.res_list[level_idx], attn, proj, fuse)
            if up is not None: x = up(x)

        return self.out(x).tanh()

# =========================
# Diffusion Schedule & DDIM Sampler (x0 prediction)
# =========================

class DiffusionSchedule:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=2e-2, device="cuda"):
        self.T = T
        betas = torch.linspace(beta_start, beta_end, T, device=device)
        alphas = 1.0 - betas
        alpha_bar = torch.cumprod(alphas, dim=0)
        self.register(betas, alphas, alpha_bar)
    def register(self, betas, alphas, alpha_bar):
        self.betas = betas; self.alphas = alphas; self.alpha_bar = alpha_bar
        self.sqrt_ab = torch.sqrt(alpha_bar); self.sqrt_1m_ab = torch.sqrt(1.0 - alpha_bar)
        self.sqrt_a = torch.sqrt(alphas); self.inv_sqrt_a = 1.0/torch.sqrt(alphas)
    def sample_q(self, x0, t, eps=None):
        B = x0.size(0)
        if eps is None: eps = torch.randn_like(x0)
        sqrt_ab_t = self.sqrt_ab[t].view(B,1,1,1)
        sqrt_1m_ab_t = self.sqrt_1m_ab[t].view(B,1,1,1)
        x_t = sqrt_ab_t * x0 + sqrt_1m_ab_t * eps
        return x_t, eps

@torch.no_grad()
def sample_ddim_whole(model, sched, cond, steps=50, eta=0.0):
    device = cond.device
    B, _, H, W = cond.shape
    T = sched.T
    ts = torch.linspace(T-1, 0, steps, device=device).long()
    x = torch.randn(B, 3, H, W, device=device)

    for i, t in enumerate(ts):
        tt = torch.full((B,), int(t.item()), device=device, dtype=torch.long)
        pred_x0 = model(x, tt, cond)
        ab_t    = sched.alpha_bar[tt].view(B,1,1,1)
        eps_t   = (x - torch.sqrt(ab_t)*pred_x0) / torch.sqrt(1 - ab_t + 1e-8)

        if i == steps - 1:
            x = pred_x0
            break

        t_next  = ts[min(i+1, steps-1)]
        tt_next = torch.full((B,), int(t_next.item()), device=device, dtype=torch.long)
        ab_next = sched.alpha_bar[tt_next].view(B,1,1,1)

        eta_tensor = torch.tensor(eta, device=device).view(1,1,1,1)
        sigma_t = eta_tensor * torch.sqrt((1 - ab_next)/(1 - ab_t + 1e-8) * (1 - ab_t/ab_next + 1e-8))
        noise = torch.randn_like(x)
        x = torch.sqrt(ab_next)*pred_x0 + torch.sqrt(1 - ab_next - sigma_t**2)*eps_t + sigma_t*noise

    return x.clamp(-1, 1)

@torch.no_grad()
def sample_ddim(model, sched, cond, mask, steps=50, eta=0.0):
    device = cond.device
    B, _, H, W = cond.shape
    T = sched.T
    ts = torch.linspace(T-1, 0, steps, device=device).long()

    x = torch.randn(B, 3, H, W, device=device) * mask + cond * (1 - mask)

    for i, t in enumerate(ts):
        tt = torch.full((B,), int(t.item()), device=device, dtype=torch.long)
        pred_x0 = model(x, tt, cond)
        # pred_x0 = model(x, tt, torch.cat([cond, mask], dim=1))

        ab_t  = sched.alpha_bar[tt].view(B,1,1,1)
        eps_t = (x - torch.sqrt(ab_t)*pred_x0) / torch.sqrt(1 - ab_t + 1e-8)

        if i == steps - 1:
            x = pred_x0
        else:
            t_next  = ts[i+1]
            tt_next = torch.full((B,), int(t_next.item()), device=device, dtype=torch.long)
            ab_next = sched.alpha_bar[tt_next].view(B,1,1,1)

            eta_tensor = torch.tensor(eta, device=device).view(1,1,1,1)
            sigma_t = eta_tensor * torch.sqrt(
                (1 - ab_next)/(1 - ab_t + 1e-8) * (1 - ab_t/ab_next + 1e-8)
            )
            noise = torch.randn_like(x)

            x = torch.sqrt(ab_next)*pred_x0 \
                + torch.sqrt(1 - ab_next - sigma_t**2)*eps_t \
                + sigma_t*noise

        x = x * mask + cond * (1 - mask)

    return x.clamp(-1, 1)

# =========================
# Visualization (masked inpaint view)
# =========================
@torch.no_grad()
def save_eval_grid(model, sched, loader, save_path, steps=50, eta=0.0, max_vis=8):
    model.eval()
    try:
        masked, x0, bbox, mask = next(iter(loader))
    except StopIteration:
        print("[WARN] Empty dataloader, skip visualization")
        return

    device = next(model.parameters()).device
    masked = masked.to(device)
    x0     = x0.to(device)
    mask   = mask.to(device)  # [B,1,H,W]

    vis_n = min(masked.shape[0], max_vis)
    cond_vis  = masked[:vis_n]               # 条件：被遮挡图
    x0_vis    = x0[:vis_n]                   # GT
    mask_vis  = mask[:vis_n]                 # [B,1,H,W]

    # 用条件图进行 DDIM 采样（模型会给出完整图）
    pred_full = sample_ddim(model, sched, cond_vis, mask=mask, steps=steps, eta=eta)  # [-1,1]

    # 只在遮罩区域替换，未遮罩区域直接保留原上下文
    comp = cond_vis * (1 - mask_vis) + pred_full * mask_vis  # inpaint 复合结果

    # 拼图：cond / GT / comp
    grid = vutils.make_grid(torch.cat([cond_vis, x0_vis, comp], dim=0), nrow=vis_n)
    img  = to_uint8(grid.permute(1,2,0).cpu()).numpy()
    Image.fromarray(img).save(save_path)




In [ ]:
# =========================
# Train Loop (pixel-space x0 prediction, masked diffusion & masked loss)
# =========================

def train(
    train_root="drive/MyDrive/resize1",
    val_root="drive/MyDrive/resizeval1",
    image_size=512,
    batch_size=1,
    lr=1e-5,
    epochs=50,
    num_workers=4,
    T=400,
    save_dir="runs/outpaint_x0_512",
    ema_decay=0.999,
    width_scale=1.25,
    blocks_per_stage=3,
    attn_res=(64,),
    highres_res=(128,256,512),
):
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] device = {device}")

    # Datasets
    train_set = OutpaintDataset(train_root, img_size=image_size, masks_per_image=40)
    val_set   = OutpaintDataset(val_root,   img_size=image_size, masks_per_image=8)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True, drop_last=False)

    # Model (pixel space)
    model = ConditionalUNet(
        base=128, ch_mult=(1,2,4,8), width_scale=width_scale,
        blocks_per_stage=blocks_per_stage, time_dim=320,
        in_channels=6, out_channels=3,  # [x_t(3) + cond(3)]
        in_res=image_size,
        attn_res=set(attn_res),
        highres_res=set(highres_res),
        attn_heads=4, attn_dropout=0.0
    ).to(device)

    # EMA
    model_ema = ConditionalUNet(
        base=128, ch_mult=(1,2,4,8), width_scale=width_scale,
        blocks_per_stage=blocks_per_stage, time_dim=320,
        in_channels=6, out_channels=3,
        in_res=image_size,
        attn_res=set(attn_res),
        highres_res=set(highres_res),
        attn_heads=4, attn_dropout=0.0
    ).to(device)
    model_ema.load_state_dict(model.state_dict())
    for p in model_ema.parameters():
        p.requires_grad_(False)

    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = DiffusionSchedule(T=T, device=device)

    def masked_recon_loss(pred, target, mask):
        """
        只在 mask 区域计算 0.4*L1 + 0.6*MSE
        pred/target: [B,3,H,W], mask: [B,1,H,W]
        """
        m = mask.expand(-1, pred.size(1), -1, -1)  # -> [B,3,H,W]
        l1 = (pred - target).abs() * m
        l2 = (pred - target).pow(2) * m
        denom = m.sum() + 1e-8
        return 0.4 * l1.sum() / denom + 0.6 * l2.sum() / denom

    best_val = float('inf')

    for epoch in range(1, epochs+1):
        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for masked, x0, bbox, mask in pbar:
            masked = masked.to(device); x0 = x0.to(device)
            mask = mask.to(device)  # [B,1,H,W]
            B, C, H, W = x0.shape

            # === 只在 mask 内扩散加噪，其余保持原图 ===
            t_max = int(0.3 * T) if epoch < 1 else int(0.7 * T) if epoch < 2 else T
            t = torch.randint(0, t_max, (B,), device=device)
            x_t_full, _eps = sched.sample_q(x0, t)            # 全图噪声建议
            x_t = x0 * (1 - mask) + x_t_full * mask           # 只替换遮挡区域

            # 条件：被遮挡的图（3通道）；输入拼接在模型里已处理为 [x_t, cond]
            pred_x0 = model(x_t, t, masked)

            # === 只在 mask 区域计算损失 ===
            loss = masked_recon_loss(pred_x0, x0, mask)

            optim.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

            # EMA
            with torch.no_grad():
                for p, q in zip(model.parameters(), model_ema.parameters()):
                    q.data.mul_(ema_decay).add_(p.data, alpha=1-ema_decay)

            pbar.set_postfix(loss=f"{float(loss.item()):.4f}")

        # ===== Validation (EMA) =====
        model_ema.eval(); val_loss = 0.0; n_batches = 0
        with torch.no_grad():
            for masked, x0, bbox, mask in tqdm(val_loader, desc="Validating"):
                masked = masked.to(device); x0 = x0.to(device); mask = mask.to(device)
                B, C, H, W = x0.shape
                t = torch.randint(0, T, (B,), device=device).long()
                x_t_full, _ = sched.sample_q(x0, t)
                x_t = x0 * (1 - mask) + x_t_full * mask
                pred_x0 = model_ema(x_t, t, masked)
                loss = masked_recon_loss(pred_x0, x0, mask)
                val_loss += float(loss.item()); n_batches += 1
        val_loss /= max(1, n_batches)
        print(f"[VAL] epoch {epoch}: {val_loss:.4f}")

        # Visualization (EMA): 100 & 200 steps
        steps = 100
        vis_path = os.path.join(save_dir, f"vis_epoch_{epoch:03d}_s{steps}.png")
        save_eval_grid(model_ema, sched, val_loader, vis_path, steps=steps, eta=0.0, max_vis=8)
        steps = 200
        vis_path = os.path.join(save_dir, f"vis_epoch_{epoch:03d}_s{steps}.png")
        save_eval_grid(model_ema, sched, val_loader, vis_path, steps=steps, eta=0.0, max_vis=8)

        # Save best
        if val_loss < best_val:
            best_val = val_loss
            torch.save({
                'model': model.state_dict(),
                'ema': model_ema.state_dict(),
                'opt': optim.state_dict(),
                'epoch': epoch,
            }, os.path.join(save_dir, 'train.pt'))


In [ ]:
if __name__ == "__main__":
    train()

Output hidden; open in https://colab.research.google.com to view.